# Solution B2 · A single GridSearchCV to choose between different models

**Type**: Implement with scikit-learn

## Objective

Use a `Pipeline` with a placeholder step and a **list of dictionaries** grid to search at the same time between two model families
(logistic regression and random forest) with their own hyperparameters, and read `cv_results_` critically.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = ds.titanic()
X, y = df.drop(columns="survived"), df["survived"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
NUM = ["age", "siblings_spouses", "parents_children", "fare"]
CAT = ["sex", "port", "deck", "title"]
cv = StratifiedKFold(5, shuffle=True, random_state=0)

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

prep = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUM),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), CAT),
    ("pclass", "passthrough", ["pclass"]),
], sparse_threshold=0)

## Commented solution

In [ ]:
pipe = Pipeline([("prep", prep), ("clf", LogisticRegression())])
grid = [
    {"clf": [LogisticRegression(max_iter=2000)], "clf__C": [0.01, 0.1, 1, 10]},
    {"clf": [RandomForestClassifier(random_state=0)], "clf__max_depth": [3, 6, None], "clf__min_samples_leaf": [1, 5]},
]

In [ ]:
search = GridSearchCV(pipe, grid, cv=cv, scoring="roc_auc").fit(X_tr, y_tr)
n_candidates = len(search.cv_results_["params"])
n_trainings = n_candidates * cv.get_n_splits() + 1
best_family = type(search.best_estimator_[-1]).__name__
test_auc = roc_auc_score(y_te, search.predict_proba(X_te)[:, 1])
print(f"{n_candidates} candidates, {n_trainings} trainings | best: {best_family} {search.best_params_}")
print(f"best_score_ {search.best_score_:.3f} | test AUC {test_auc:.3f}")

In [ ]:
r = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
gap = r["mean_test_score"].iloc[0] - r["mean_test_score"].iloc[1]
best_std = r["std_test_score"].iloc[0]
significant_difference = bool(gap > best_std)
print(r[["param_clf", "mean_test_score", "std_test_score", "rank_test_score"]].head(5).round(4).to_string())
print(f"gap between first and second: {gap:.4f} | standard deviation of the best: {best_std:.4f}")

## Check

In [ ]:
with verify():
    check(isinstance(grid, list) and len(grid) == 2, "grid = list of two dictionaries", "one dictionary per model family")
    check(n_candidates == 4 + 3 * 2 and n_trainings == 10 * 5 + 1, "10 candidates (4 + 6) and 51 trainings", "candidates x folds + refit")
    check(best_family in ("LogisticRegression", "RandomForestClassifier") and best_family == type(search.best_estimator_[-1]).__name__,
          "best_family consistent with best_estimator_", "use type(search.best_estimator_[-1]).__name__")
    check(0.80 < test_auc < 0.93, "reasonable test AUC", "test_auc = roc_auc_score(y_te, search.predict_proba(X_te)[:, 1])")
    check(close(gap, r["mean_test_score"].iloc[0] - r["mean_test_score"].iloc[1]) and close(best_std, r["std_test_score"].iloc[0]),
          "gap and standard deviation of the best are correct", "first two rows of r, sorted by rank")
    check(significant_difference == (gap > best_std), "judgment consistent with the numbers", "significant_difference = gap > best_std")
    check(gap < 0.02, "the gap between the first two is small (less than 0.02): the choice is between nearly equivalent candidates",
          "look at mean_test_score of the first two")
done()

## What you learned

- With `clf` as a placeholder step and a **list of dictionaries** you can compare model families with different hyperparameters in a
  single search, with the same preprocessing and the same folds: it is a fair comparison.
- The count of trainings (10 candidates × 5 folds + 1) is what you should do **before** launching the search.
- The gap between the first and the second ranked is much smaller than the standard deviation across folds: the ranking says
  little. When the candidates are statistically indistinguishable you choose the simplest and the fastest (the standard error rule,
  exercise D1).

## Common pitfall

Forgetting `refit` with several metrics: if `scoring` is a dictionary, `refit` must say **which** metric to use to choose the
`best_estimator_`. Without it, the search works but does not produce the final model.

## Interview question

> *Click on a question to show its answer.*

<details><summary>How do you fairly compare a random forest and a logistic regression on a dataset with common preprocessing?</summary>

With the same preprocessing pipeline, the same folds and a single `GridSearchCV` with a list of grids (a placeholder step for the model),
so every family has its own hyperparameters optimized with the same cross-validation budget. Then I evaluate the best on the test set
once, and look at whether the difference from the second exceeds the variability across folds.

</details>